In [2]:
# Chipathon 2026 - NYCMOS
# This notebook computes the sizing for Gilbert cell mixer switching pair & LO quad based on gm/Id method
# Resource: https://github.com/bmurmann/Chipathon2025/tree/main

from pygmid import Lookup as lk
from scipy.io import loadmat
import numpy as np
import pandas as pd

In [3]:
# Change file depending on where you cloned repo
n = lk('nfet_03v3.mat')
print(n._Lookup__DATA.keys())

dict_keys(['INFO', 'CORNER', 'TEMP', 'VGS', 'VDS', 'VSB', 'L', 'W', 'NFING', 'ID', 'VT', 'GM', 'GMB', 'GDS', 'CGG', 'CGB', 'CGD', 'CGS', 'CDD', 'CSS', 'STH', 'SFL'])


In [7]:
# Target
gain_db = np.array([5, 10, 15])
Av = 10**(gain_db/20)

rd = 2e3 # First pass
gm_required = Av / (2/np.pi * rd)

gm_id = np.array([10, 10, 10]) #moderate inversion
l = 0.28

In [8]:
# Sizing for transconductance stage

id_per_side = gm_required / gm_id
Jd = n.lookup('ID_W', GM_ID=gm_id, L = l)
W = id_per_side / Jd
cgg_w = n.lookup('CGG_W', GM_ID=gm_id, L = l)

cgg = cgg_w * W
ft = gm_required / (2 * np.pi * cgg) # Estimate transit freq

vdsat_est = 2/gm_id
itail = 2 * id_per_side

vgs = n.lookup('VGS', GM_ID=gm_id, L=l, VDS=1.65)

df = pd.DataFrame(
    [gain_db, Av, gm_required, gm_id, id_per_side, itail, Jd, W, cgg, ft, vdsat_est],
    index=['gain_db', 'Av', 'gm_required', 'gm/id', 'id_per_side (uA)', 'itail', 'ID_W', 'W_um', 'Cgg', 'ft', 'Vdsat'],
    columns = ['option1', 'option2', 'option3']
)
df

,option1,option2,option3
gain_db,5.000000e+00,1.000000e+01,1.500000e+01
Av,1.778279e+00,3.162278e+00,5.623413e+00
gm_required,1.396657e-03,2.483647e-03,4.416618e-03
gm/id,1.000000e+01,1.000000e+01,1.000000e+01
id_per_side (uA),1.396657e-04,2.483647e-04,4.416618e-04
itail,2.793315e-04,4.967294e-04,8.833237e-04
ID_W,6.847242e-06,6.847242e-06,6.847242e-06
W_um,2.039737e+01,3.627223e+01,6.450216e+01
Cgg,2.240166e-14,3.983641e-14,7.084027e-14
ft,9.922699e+09,9.922699e+09,9.922699e+09


In [9]:
# RF switching quad
# Size based on current it must steer
rf_option = 1; 

gm_id_ss = np.array([8, 10, 12, 15, 18])
l_ss = 0.28

id_ss = id_per_side[rf_option]
jd_ss = n.lookup('ID_W', GM_ID=gm_id_ss, L=l_ss)
w_ss = id_ss / jd_ss

cgg_w_ss = n.lookup('CGG_W', GM_ID=gm_id_ss, L=l_ss)
cg_ss = w_ss * cgg_w_ss

gm_ss = gm_id_ss * id_ss
ft_ss = gm_ss / (2*np.pi*cg_ss)

vdsat_ss = 2/gm_id_ss

df_ss = pd.DataFrame(
    [gm_id_ss, id_ss*np.ones_like(gm_id_ss), w_ss, gm_ss, ft_ss, vdsat_ss],
    index=['gm_id', 'id_ss', 'w_ss', 'gm_ss', 'ft_ss', 'vdsat_ss'])
df_ss


,0,1,2,3,4
gm_id,8.000000e+00,1.000000e+01,1.200000e+01,1.500000e+01,1.800000e+01
id_ss,2.483647e-04,2.483647e-04,2.483647e-04,2.483647e-04,2.483647e-04
w_ss,2.239303e+01,3.627223e+01,5.781187e+01,1.184109e+02,2.656923e+02
gm_ss,1.986918e-03,2.483647e-03,2.980376e-03,3.725471e-03,4.470565e-03
ft_ss,1.251254e+10,9.922699e+09,7.678494e+09,4.891709e+09,2.745315e+09
vdsat_ss,2.500000e-01,2.000000e-01,1.666667e-01,1.333333e-01,1.111111e-01
